# Sagemaker Coder V2 into V3

The aim of this notebook is to showcase the difference between old and new models regarding their cutoff date.<br>
I will try to translate a piece of code containing Sagemaker SDK V2 to a piece code using Sagemaker SDK V3.<br>
The cutoff point of the old model will be different than the cutoff date of the new one.<br>
<br>
The new one will be able to answer the question correctly, while the old one will at best rewrite the code<br>
in the old version of the SDK.

In [ ]:
# imports
import os
from dotenv import load_dotenv
from openai import OpenAI
from src.code_snippet import pythoncode
import gradio as gr
from IPython.display import Markdown, display

In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [13]:

system_prompt = f"""
Your task is to convert code containing the SageMaker Python SDK V2 into code containing the SageMaker Python SDK V3.
Respond only with code containing the SageMaker Python SDK V3. Do not provide any explanation other than occasional comments.
"""

def user_prompt_for(python):
    return f"""
Port this SageMaker Python SDK V2 code into SageMaker Python SDK V3 code with the implementation that produces identical output.
Your response will be written to a file called main. Respond only with code.
Code to port:

```python
{python}
```
"""

In [14]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [4]:
openai = OpenAI()

In [27]:
models = ['gpt-4.1-mini', 'gpt-5.6-luna']

In [29]:
def translate(snippet, model):
    response = openai.chat.completions.create(model=model, messages=messages_for(snippet))
    reply = response.choices[0].message.content
    return reply

In [30]:
with gr.Blocks() as ui:
    with gr.Row():
        v2 = gr.Textbox(label="V2 code:", lines=28, value=pythoncode)
        v3 = gr.Textbox(label="V3 code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(translate, inputs=[v2, model], outputs=[v3])

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
